<a href="https://colab.research.google.com/github/leeknnn/viktoriiashkorubska-41/blob/master/lab_work_8/lab_work_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd

np.random.seed(42)
base_temp = 9.5
amplitude = 14
city = "Київ"
rows = []

for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)

print("--- Завдання 1: Групбай за роком ---")
year_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
display(year_stats)

print("\n--- Завдання 2: Групбай за місяцем ---")
month_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])
display(month_stats)

print("\n--- Завдання 3: pivot_table ---")
pivot_t = climate.pivot_table(index="місяць", columns="рік", values="температура", aggfunc="mean")
display(pivot_t)

print("\n--- Завдання 4: Похідні категорії і crosstab ---")
def get_season(m):
    if m in [12, 1, 2]:
        return "зима"
    elif m in [3, 4, 5]:
        return "весна"
    elif m in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

climate["сезон"] = climate["місяць"].apply(get_season)
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

cross = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
display(cross)

print("\n--- Завдання 5: pivot без aggfunc ---")
pivot_simple = climate.pivot(index="місяць", columns="рік", values="температура")
display(pivot_simple.head())

--- Завдання 1: Групбай за роком ---


,mean,min,max
рік,,,
2021,9.783333,-4.0,25.1
2022,8.916667,-4.5,22.6
2023,9.308333,-5.0,23.5
2024,9.158333,-4.6,23.4



--- Завдання 2: Групбай за місяцем ---


,mean,std
місяць,,
1,-4.400,0.424264
2,-3.600,1.104536
3,1.600,1.023067
4,9.875,0.865544
5,16.225,0.727438
6,21.600,0.294392
7,23.500,1.116542
8,21.850,1.420094
9,16.375,1.250000



--- Завдання 3: pivot_table ---


рік,2021,2022,2023,2024
місяць,,,,
1,-4.0,-4.3,-5.0,-4.3
2,-2.8,-4.5,-2.5,-4.6
3,3.1,0.8,1.3,1.2
4,11.0,8.9,9.9,9.7
5,16.3,15.5,15.9,17.2
6,21.4,21.9,21.3,21.8
7,25.1,22.6,22.9,23.4
8,22.4,20.2,23.5,21.3
9,16.0,18.0,16.5,15.0



--- Завдання 4: Похідні категорії і crosstab ---


тепліше_за_середнє,False,True
сезон,,
весна,5,7
зима,12,0
літо,0,12
осінь,7,5



--- Завдання 5: pivot без aggfunc ---


рік,2021,2022,2023,2024
місяць,,,,
1,-4.0,-4.3,-5.0,-4.3
2,-2.8,-4.5,-2.5,-4.6
3,3.1,0.8,1.3,1.2
4,11.0,8.9,9.9,9.7
5,16.3,15.5,15.9,17.2


## Відповіді на контрольні питання

1. **Яка логіка split-apply-combine виконується через groupby()?**
   - На першому етапі (split) набір даних розбивається на окремі групи відповідно до умов групування. На другому етапі (apply) до кожної створеної групи незалежно застосовується вказана агрегувальна функція або обчислення. На третьому етапі (combine) результати обробки всіх груп об'єднуються в єдину результуючу структуру даних.

2. **Чому pivot() завершується помилкою на дублікатах комбінацій індекс/стовпець, а pivot_table() ні?**
   - Метод pivot() призначений для чистого перетворення форми і вимагає унікальності кожної пари комбінацій значень індексу та стовпця; якщо на одну комірку припадає кілька рядків, він не знає, яке з них обрати, і викликає ValueError. Метод pivot_table() містить вбудований параметр aggfunc, який автоматично агрегує кілька значень в одне єдине число.

3. **Що показує crosstab() і чим він відрізняється від groupby(...).size()?**
   - Функція crosstab() створює матрицю перехресного розподілу частот для двох або більше категоріальних змінних, автоматично підраховуючи кількість комбінацій. На відміну від groupby з методом size(), який видає довгий мультиіндексний рядок або серію результатів, crosstab формує зручну двовимірну таблицю згортання.

4. **Чому дані для groupby і pivot_table мають бути в довгому (tidy) форматі?**
   - У довгому форматі кожна змінна міститься в окремому стовпці, а кожне спостереження у новому рядку, що дозволяє бібліотеці Pandas чітко і однозначно ідентифікувати вимірювання за ключами. Широкі формати розтягують значення по заголовках стовпців, через що стандартні групування та агрегації стають неможливими без попередньої трансформації.